<a href="https://colab.research.google.com/github/Zyu-Peng/learning_code/blob/AF2/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.0: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [39]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

In [40]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/input_fasta' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/result' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}

import sys
import re
from pathlib import Path

from colabfold.batch import run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging

# 保留原有pdbfixer路径修复逻辑
python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

# 核心修复：返回4元组格式（适配ColabFold的run函数要求）
def parse_all_sequences(input_path):
    queries = []
    input_path = Path(input_path)

    # 处理目录/文件输入
    if input_path.is_dir():
        fasta_files = list(input_path.glob("*.fasta")) + list(input_path.glob("*.fa"))
        if not fasta_files:
            raise FileNotFoundError(f"目录 {input_path} 下无FASTA文件")
    else:
        fasta_files = [input_path]

    # 解析所有FASTA文件的所有序列
    for fasta_file in fasta_files:
        with open(fasta_file, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]

        current_name, current_seq = None, []
        for line in lines:
            if line.startswith('>'):
                if current_name and current_seq:
                    # 关键修复：返回4元组 (序列名, 序列, None, None)
                    queries.append((current_name.split('|')[0].strip(), ''.join(current_seq), None, None))
                current_name = line[1:]
                current_seq = []
            else:
                current_seq.append(re.sub(r'[^A-Za-z]', '', line).upper())
        if current_name and current_seq:
            queries.append((current_name.split('|')[0].strip(), ''.join(current_seq), None, None))

    return queries, False

# 保留原有日志设置
setup_logging(Path(result_dir).joinpath("log.txt"))

# 替换原有get_queries调用，改为读取所有序列
queries, is_complex = parse_all_sequences(input_dir)

# 保留原有run函数调用（参数完全不变）
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    num_relax=num_relax,
    relax_max_iterations=relax_max_iterations,
    msa_mode=msa_mode,
    model_type="auto",
    num_models=num_models,
    num_recycles=num_recycles,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=default_data_dir,
    keep_existing_results=do_not_overwrite_results,
    rank_by="auto",
    pair_mode="unpaired+paired",
    stop_at_score=stop_at_score,
    zip_results=zip_results,
    user_agent="colabfold/google-colab-batch",
)

2026-03-16 14:01:21,426 Running on GPU
2026-03-16 14:01:21,442 Found 5 citations for tools or databases
2026-03-16 14:01:21,443 Query 1/966: seq_1 (length 40)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-16 14:01:29,872 Padding length to 50
2026-03-16 14:01:55,909 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=57.4 pTM=0.308
2026-03-16 14:02:20,800 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=77.6 pTM=0.498 tol=1.75
2026-03-16 14:02:25,418 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=81.9 pTM=0.538 tol=0.349
2026-03-16 14:02:30,036 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=82.4 pTM=0.541 tol=0.15
2026-03-16 14:02:30,037 alphafold2_ptm_model_1_seed_000 took 60.2s (3 recycles)
2026-03-16 14:02:34,677 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=52.1 pTM=0.237
2026-03-16 14:02:39,307 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=52 pTM=0.247 tol=0.856
2026-03-16 14:02:43,920 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=53.1 pTM=0.261 tol=3.07
2026-03-16 14:02:48,499 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=49.5 pTM=0.235 tol=1.42
2026-03-16 14:02:48,500 alphafold2_ptm_model_2_seed_000 took 18.4s (3 recycles)
2026-03-16 14:02:53,075 alphafold2_ptm_model_

PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

2026-03-16 14:03:44,719 Sleeping for 5s. Reason: PENDING


...

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:18 remaining: 00:00]


2026-03-16 14:07:34,346 Padding length to 50
2026-03-16 14:07:38,929 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=65.4 pTM=0.307
2026-03-16 14:07:43,385 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=69.2 pTM=0.32 tol=1.12
2026-03-16 14:07:47,875 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=70.1 pTM=0.323 tol=0.284
2026-03-16 14:07:52,391 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=69.8 pTM=0.321 tol=0.0795
2026-03-16 14:07:52,391 alphafold2_ptm_model_1_seed_000 took 18.0s (3 recycles)
2026-03-16 14:07:56,987 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=57.8 pTM=0.241
2026-03-16 14:08:01,587 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=56.9 pTM=0.219 tol=1.04
2026-03-16 14:08:06,201 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=56.2 pTM=0.193 tol=0.923
2026-03-16 14:08:10,775 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=55.4 pTM=0.181 tol=0.649
2026-03-16 14:08:10,776 alphafold2_ptm_model_2_seed_000 took 18.4s (3 recycles)
2026-03-16 14:08:15,349 alphafold2_ptm_mo

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-16 14:09:08,434 Padding length to 50
2026-03-16 14:09:13,083 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=90 pTM=0.479
2026-03-16 14:09:17,587 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=91.9 pTM=0.499 tol=0.242
2026-03-16 14:09:22,108 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=92.6 pTM=0.502 tol=0.0823
2026-03-16 14:09:26,675 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=92.3 pTM=0.499 tol=0.0573
2026-03-16 14:09:26,676 alphafold2_ptm_model_1_seed_000 took 18.2s (3 recycles)
2026-03-16 14:09:31,314 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=84.4 pTM=0.448
2026-03-16 14:09:35,928 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=89 pTM=0.474 tol=0.24
2026-03-16 14:09:40,509 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=90 pTM=0.479 tol=0.0504
2026-03-16 14:09:45,060 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=89.6 pTM=0.473 tol=0.0588
2026-03-16 14:09:45,060 alphafold2_ptm_model_2_seed_000 took 18.4s (3 recycles)
2026-03-16 14:09:49,639 alphafold2_ptm_mod

KeyboardInterrupt: 